In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_CSV = Path(r'D:\tcc_ibov\data\raw\bvsp_stooq.csv')


In [2]:
RAW_CSV = Path('/content/drive/MyDrive/tcc_ibov/data/raw/bvsp_stooq.csv')
raw = pd.read_csv(RAW_CSV, parse_dates=['Date'], index_col='Date').sort_index()

print(raw.shape, '|', raw.index.min(), '->', raw.index.max())
print("Colunas:", raw.columns.tolist())
print("Duplicatas no índice:", raw.index.duplicated().sum())
print("\nNulos por coluna:")
print(raw.isna().sum())

study = raw.loc['2019-01-01':'2024-12-31']
print(f"\nPregões 2019-2024: {len(study)}")
print(study.groupby(study.index.year).size())

print(f"\nLinhas em sábado/domingo: {(raw.index.dayofweek >= 5).sum()}")

anchors = {'2020-03-23': 63570, '2024-12-30': 120283}
for date, expected in anchors.items():
    if date in raw.index:
        got = float(raw.loc[date, 'Close'])
        dev = abs(got - expected) / expected
        print(f"{date}: esperado ~{expected:,} | obtido {got:,.0f} | "
              f"desvio {dev:.2%} [{'OK' if dev < 0.01 else 'FALHOU'}]")
    else:
        print(f"{date}: DATA AUSENTE — investigar")

ret = raw['Close'].pct_change()
print(f"\nMaiores quedas:\n{ret.nsmallest(3)}")
print(f"\nMaiores altas:\n{ret.nlargest(3)}")

print(f"\nDias com High < Low: {(raw['High'] < raw['Low']).sum()}")
if 'Volume' in raw.columns:
    v = raw['Volume']
    print(f"Volume — zeros: {(v == 0).sum()}/{len(v)} | nulos: {v.isna().sum()}")

(1615, 5) | 2018-07-02 00:00:00 -> 2024-12-30 00:00:00
Colunas: ['Open', 'High', 'Low', 'Close', 'Volume']
Duplicatas no índice: 0

Nulos por coluna:
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Pregões 2019-2024: 1493
Date
2019    248
2020    249
2021    247
2022    250
2023    248
2024    251
dtype: int64

Linhas em sábado/domingo: 0
2020-03-23: esperado ~63,570 | obtido 63,570 | desvio 0.00% [OK]
2024-12-30: esperado ~120,283 | obtido 120,283 | desvio 0.00% [OK]

Maiores quedas:
Date
2020-03-12   -0.147804
2020-03-16   -0.139213
2020-03-09   -0.121735
Name: Close, dtype: float64

Maiores altas:
Date
2020-03-13    0.139089
2020-03-24    0.096897
2020-03-25    0.074951
Name: Close, dtype: float64

Dias com High < Low: 0
Volume — zeros: 0/1615 | nulos: 0


In [3]:
print("Mediana de volume por ano:")
print(raw['Volume'].groupby(raw.index.year).median())
print("\nVolume — min/max:", raw['Volume'].min(), '/', raw['Volume'].max())

clean = raw.copy()
clean['log_ret'] = np.log(clean['Close'] / clean['Close'].shift(1))

print(f"\nNaNs em log_ret: {clean['log_ret'].isna().sum()} (esperado: 1, na 1a linha)")

lr = clean['log_ret'].dropna()
print(f"\nlog_ret — média diária: {lr.mean():+.5f} | desvio: {lr.std():.5f}")
print(f"anualizado — retorno: {lr.mean()*252:+.2%} | vol: {lr.std()*np.sqrt(252):.2%}")
print(f"assimetria: {lr.skew():+.2f} | curtose (excesso): {lr.kurt():.2f}")

Mediana de volume por ano:
Date
2018    3.974419e+08
2019    4.745756e+08
2020    9.522846e+08
2021    9.445801e+08
2022    1.269244e+09
2023    1.171625e+09
2024    9.289240e+08
Name: Volume, dtype: float64

Volume — min/max: 171589792 / 2602931500

NaNs em log_ret: 1 (esperado: 1, na 1a linha)

log_ret — média diária: +0.00031 | desvio: 0.01561
anualizado — retorno: +7.83% | vol: 24.79%
assimetria: -1.41 | curtose (excesso): 20.84


In [4]:
PROCESSED_DIR = Path('/content/drive/MyDrive/tcc_ibov/data/processed/')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PARQUET = PROCESSED_DIR / 'bvsp_clean.parquet'
clean.to_parquet(CLEAN_PARQUET)

print(f"DataFrame 'clean' salvo em: {CLEAN_PARQUET}")

DataFrame 'clean' salvo em: /content/drive/MyDrive/tcc_ibov/data/processed/bvsp_clean.parquet
